In [ ]:
import kagglehub

path = kagglehub.dataset_download("sebastianpalaciob/tomato-leaf-diseases-dataset-for-object-detection")

print("Path to dataset files:", path)

c:\Users\alijo\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\alijo\.cache\kagglehub\datasets\sebastianpalaciob\tomato-leaf-diseases-dataset-for-object-detection\versions\3


In [ ]:
from pathlib import Path

root = Path(path)  
print("ROOT:", root)

def tree(d: Path, max_depth=2, prefix=""):
    if max_depth < 0: 
        return
    entries = sorted(d.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    for i, p in enumerate(entries):
        connector = "└── " if i == len(entries)-1 else "├── "
        print(prefix + connector + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir():
            tree(p, max_depth-1, prefix + ("    " if i == len(entries)-1 else "│   "))

tree(root, max_depth=2)  


ROOT: C:\Users\alijo\.cache\kagglehub\datasets\sebastianpalaciob\tomato-leaf-diseases-dataset-for-object-detection\versions\3
└── Tomato leaf diseases dataset for Object Detection/
    ├── Dataset with augmentation/
    │   ├── test/
    │   ├── train/
    │   ├── valid/
    │   ├── data.yaml
    │   ├── README.dataset.txt
    │   └── README.roboflow.txt
    ├── Dataset without augmentation/
    │   ├── test/
    │   ├── train/
    │   ├── valid/
    │   ├── data.yaml
    │   ├── README.dataset.txt
    │   └── README.roboflow.txt
    ├── README_en.txt
    └── README_es.txt


In [ ]:
import os, random, shutil, yaml
from pathlib import Path

# === EDIT THESE ===
ROOT = Path(r"C:\Users\alijo\.cache\kagglehub\datasets\sebastianpalaciob\tomato-leaf-diseases-dataset-for-object-detection\versions\3\Tomato leaf diseases dataset for Object Detection")
SRC_VARIANT = "Dataset without augmentation"  
OUT_ROOT = Path(r"C:\Users\alijo\OneDrive\Desktop\LeafminerOnly")
KEEP_BG_FRAC = 0.20   
SEED = 42
# ===================

random.seed(SEED)
src = ROOT / SRC_VARIANT
yaml_path = src / "data.yaml"

# 1) Find leafminer class index (case-insensitive)
names = []
if yaml_path.exists():
    cfg = yaml.safe_load(open(yaml_path, "r", encoding="utf-8"))
    names = cfg.get("names", [])
else:
    print(f"[WARN] {yaml_path} not found; will try default name list.")
norm = lambda s: s.strip().lower().replace("-", " ").replace("_", " ")
try:
    leaf_idx = next(i for i,n in enumerate(names) if norm(n) == "leafminer")
except StopIteration:
    # Fallback: common variants
    candidates = {"leafminer","leaf miner"}
    try:
        leaf_idx = next(i for i,n in enumerate(names) if norm(n) in candidates)
    except StopIteration:
        raise SystemExit(f"'leafminer' not found in names: {names}")

# 2) Build split directories directly from the variant folder
def split_dirs(base: Path):
    cand = {}
    for sp in ("train","valid","val","test"):
        sd = base / sp
        if (sd / "images").is_dir() and (sd / "labels").is_dir():
            cand[sp if sp!="valid" else "val"] = (sd / "images", sd / "labels")
    return cand

splits = split_dirs(src)
if not splits:
    raise SystemExit(f"No split folders with images/labels under: {src}")

# 3) Helpers
def copy_file(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

def write_label(dst: Path, lines):
    dst.parent.mkdir(parents=True, exist_ok=True)
    with open(dst, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp"}

summary = {}
for sp_out, (imgs_dir, labs_dir) in splits.items():
    out_imgs = OUT_ROOT / sp_out / "images"
    out_labs = OUT_ROOT / sp_out / "labels"
    out_imgs.mkdir(parents=True, exist_ok=True)
    out_labs.mkdir(parents=True, exist_ok=True)

    img_files = [p for p in imgs_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
    img_map = {p.stem: p for p in img_files}

    positives, backgrounds = [], []
    for stem, ip in img_map.items():
        lp = labs_dir / f"{stem}.txt"
        if not lp.exists():
            backgrounds.append(stem)
            continue
        with open(lp, "r", encoding="utf-8") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        out_lines = []
        for ln in lines:
            parts = ln.split()
            if not parts: 
                continue
            try:
                cid = int(parts[0])
            except Exception:
                continue
            if cid == leaf_idx:
                parts[0] = "0"  # remap to single class 0
                out_lines.append(" ".join(parts))
        if out_lines:
            positives.append((stem, out_lines))
        else:
            backgrounds.append(stem)

    total_pos_imgs = len(positives)
    total_bg_cands = len(backgrounds)
    k = int(round(KEEP_BG_FRAC * total_bg_cands))
    sampled_bg = set(random.sample(backgrounds, k)) if k > 0 else set()

    kept_pos = kept_bg = 0
    for stem, out_lines in positives:
        ip = img_map[stem]
        copy_file(ip, out_imgs / ip.name)
        write_label(out_labs / f"{stem}.txt", out_lines)
        kept_pos += 1

    for stem in sampled_bg:
        ip = img_map[stem]
        copy_file(ip, out_imgs / ip.name)
        write_label(out_labs / f"{stem}.txt", [])  # empty label file for background
        kept_bg += 1

    summary[sp_out] = dict(pos_kept=kept_pos, bg_kept=kept_bg,
                           pos_total=total_pos_imgs, bg_total=total_bg_cands)

# 4) Write new single-class YAML
leaf_yaml = OUT_ROOT / "leafminer.yaml"
leaf_yaml.write_text(f"""# auto-generated single-class dataset
path: {OUT_ROOT.as_posix()}
train: train/images
val: val/images
test: {('test/images' if (OUT_ROOT/'test'/'images').is_dir() else '')}

nc: 1
names: ['leafminer']
""", encoding="utf-8")

# 5) Clear any stale caches in OUT_ROOT
for p in OUT_ROOT.rglob("*.cache"):
    try: p.unlink()
    except: pass

print("Done. New dataset at:", OUT_ROOT)
print("YAML:", leaf_yaml)
for sp, s in summary.items():
    print(f"{sp}: kept {s['pos_kept']} positives (of {s['pos_total']}), kept {s['bg_kept']} backgrounds (of {s['bg_total']})")


Done. New dataset at: C:\Users\alijo\OneDrive\Desktop\LeafminerOnly
YAML: C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\leafminer.yaml
train: kept 692 positives (of 692), kept 510 backgrounds (of 2552)
val: kept 127 positives (of 127), kept 106 backgrounds (of 532)
test: kept 122 positives (of 122), kept 60 backgrounds (of 298)


In [13]:
from pathlib import Path
import yaml

YAML_PATH = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\MixedDS.yaml"

cfg = yaml.safe_load(Path(YAML_PATH).read_text(encoding="utf-8"))
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

missing_txt = []
empty_txt = []

for img_dir in cfg["train"]:
    img_dir = Path(img_dir)
    images = [p for p in img_dir.rglob("*") if p.suffix.lower() in img_exts]

    for img in images:
        # .../train/images/xxx.jpg -> .../train/labels/xxx.txt
        label = Path(str(img).replace("\\images\\", "\\labels\\").replace("/images/", "/labels/")).with_suffix(".txt")

        if not label.exists():
            missing_txt.append(img)
        else:
            if label.read_text(encoding="utf-8", errors="ignore").strip() == "":
                empty_txt.append(img)

print(f"Missing .txt labels: {len(missing_txt)}")
print(f"Empty .txt labels:   {len(empty_txt)}\n")

print("---- Examples (missing .txt) ----")
for p in missing_txt[:30]:
    print(p)

print("\n---- Examples (empty .txt) ----")
for p in empty_txt[:1000]:
    print(p)


Missing .txt labels: 0
Empty .txt labels:   513

---- Examples (missing .txt) ----

---- Examples (empty .txt) ----
C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\train\images\03b6acc7-41a6-4670-b47c-bef6d4baea18___PSU_CG-2363_JPG_jpg.rf.a4f998d074b85a9c4d301a21dbf8a6f5.jpg
C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\train\images\1-36_thumb_png_jpg.rf.23ed400e7d87a8aa6c3f8887b9bc4c51.jpg
C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\train\images\118A_Tom_Late_jpg.rf.c7c88da202a0ee96b3b07cc3edec4351.jpg
C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\train\images\128110613-56a6d3633df78cf7729070ad_jpg.rf.920144d02a0617fb617249ad63e5aa24.jpg
C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\train\images\14321627273_58d893c3e2_b_jpg.rf.77411c3348b8d5197d2ce02817fcda00.jpg
C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\train\images\1573852-Early-Blight-e1515624612559_jpg.rf.c93db9b941c4ed5bd6d91ac0907eea49.jpg
C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\train\images\20080710-early-2Bblight-2Bl

In [14]:
from pathlib import Path
import yaml

YAML_PATH = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\MixedDS.yaml"

cfg = yaml.safe_load(Path(YAML_PATH).read_text(encoding="utf-8"))
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def count_images(paths):
    total = 0
    per_path = {}
    for p in paths:
        d = Path(p)
        n = sum(1 for f in d.rglob("*") if f.suffix.lower() in img_exts)
        per_path[str(d)] = n
        total += n
    return total, per_path

for split in ["train", "val", "test"]:
    if split not in cfg:
        continue
    paths = cfg[split] if isinstance(cfg[split], list) else [cfg[split]]
    total, per_path = count_images(paths)

    print(f"\n=== {split.upper()} ===")
    print(f"Total images: {total}")
    for k, v in per_path.items():
        print(f"{v:6d}  {k}")



=== TRAIN ===
Total images: 2135
  1202  C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\train\images
   149  C:\Users\alijo\OneDrive\Desktop\Noisy tiled DS\train\images
   664  C:\Users\alijo\OneDrive\Desktop\Tomato Leaf Disease.v6i.yolov8\train\images
   120  C:\Users\alijo\OneDrive\Desktop\Tomato pest-diseases.v1-tomato_v1.yolov8\train\images

=== VAL ===
Total images: 314
   233  C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\val\images
    57  C:\Users\alijo\OneDrive\Desktop\Tomato Leaf Disease.v6i.yolov8\val\images
    14  C:\Users\alijo\OneDrive\Desktop\Noisy tiled DS\val\images
    10  C:\Users\alijo\OneDrive\Desktop\Tomato pest-diseases.v1-tomato_v1.yolov8\valid\images

=== TEST ===
Total images: 197
   182  C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\test\images
     9  C:\Users\alijo\OneDrive\Desktop\Noisy tiled DS\test\images
     6  C:\Users\alijo\OneDrive\Desktop\Tomato pest-diseases.v1-tomato_v1.yolov8\test\images


In [2]:
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path=r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\Optimization\HYB_GEOML2_BN\YOLO_V8_BN_0.7_HYB_L2GEOM_BN_PRUNE\step_0_finetune\weights\best_saved_model\best_full_integer_quant.tflite")
interpreter.allocate_tensors()
for t in interpreter.get_tensor_details():
    q = t.get("quantization_parameters", {})
    scales = q.get("scales", [])
    zero_points = q.get("zero_points", [])
    if len(scales):
        print(t["name"], t["dtype"], "scales:", len(scales), "zp unique:", set(zero_points.tolist() if hasattr(zero_points, "tolist") else zero_points))


serving_default_images:0 <class 'numpy.int8'> scales: 1 zp unique: {-128}
tfl.pseudo_qconst <class 'numpy.int8'> scales: 1 zp unique: {-128}
tfl.pseudo_qconst1 <class 'numpy.int8'> scales: 1 zp unique: {-128}
tfl.pseudo_qconst2 <class 'numpy.int8'> scales: 1 zp unique: {-128}
tfl.pseudo_qconst3 <class 'numpy.int32'> scales: 1 zp unique: {0}
tfl.pseudo_qconst4 <class 'numpy.int8'> scales: 1 zp unique: {0}
tfl.pseudo_qconst5 <class 'numpy.int32'> scales: 1 zp unique: {0}
tfl.pseudo_qconst6 <class 'numpy.int8'> scales: 1 zp unique: {0}
tfl.pseudo_qconst7 <class 'numpy.int32'> scales: 64 zp unique: {0}
tfl.pseudo_qconst8 <class 'numpy.int8'> scales: 64 zp unique: {0}
tfl.pseudo_qconst9 <class 'numpy.int32'> scales: 64 zp unique: {0}
tfl.pseudo_qconst10 <class 'numpy.int8'> scales: 64 zp unique: {0}
tfl.pseudo_qconst11 <class 'numpy.int32'> scales: 64 zp unique: {0}
tfl.pseudo_qconst12 <class 'numpy.int8'> scales: 64 zp unique: {0}
tfl.pseudo_qconst13 <class 'numpy.int32'> scales: 64 zp uni

c:\Users\alijo\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
#code

In [2]:
from PIL import Image

img = Image.open(r"C:\Users\alijo\OneDrive\Desktop\BachelorThesisCode\BachelorThesis\Tomato pest-diseases.v1-tomato_v1.yolov8\test\images\leaf-miner-damage-01312017-min_jpg.rf.299a7c0e0dcf403233d6adc35dd80104.jpg")
width, height = img.size
print(f"Image size: {width}x{height} pixels")

Image size: 640x640 pixels
